# पाठ ०१ - एआई एजेन्टहरूको परिचय

**शुरुआतीहरूका लागि एआई एजेन्टहरू** कोर्सको पहिलो पाठमा स्वागत छ!

एक **एआई एजेन्ट** यस्तो प्रोग्राम हो जसले ठूलो भाषा मोडेल (LLM) लाई आफ्नो तर्कशक्ति इन्जिनको रूपमा प्रयोग गर्छ र वास्तविक संसारमा *कारबाहीहरू* लिन सक्छ — एपीआईहरू कल गर्न, डाटाबेसहरू सोध्न, वा कोड चलाउन — प्रयोगकर्ताको तर्फबाट लक्ष्य प्राप्त गर्न।

यस नोटबुकमा तपाईंले आफ्नो पहिलो एजेन्ट बनाउनु हुनेछ: एक **यात्रा एजेन्ट** जुन छुट्टी गन्तव्यहरूको सिफारिस गर्दछ। यस क्रममा तपाईंले सिक्नेछ:

1. **Microsoft Agent Framework** प्रयोग गरी Azure AI Foundry Agent Service सँग जडान गर्ने।
2. एजेन्टलाई एक **उपकरण** दिने — एक साधा Python फङ्सन जसलाई यसले कल गर्न सक्छ।
3. एजेन्ट चलाउने र यसको प्रतिक्रिया निरीक्षण गर्ने।
4. एजेन्टको प्रतिक्रियालाई टोकन-द्वारा-टोकन स्ट्रिम गर्ने।


## सेटअप

यस नोटबुक चलाउनु अघि, सुनिश्चित गर्नुहोस् कि तपाईंसँग:

1. **एउटा Azure AI Foundry प्रोजेक्ट** जसमा एक डिप्लोय गरिएको च्याट मोडेल छ (जस्तै `gpt-4o-mini`)।
2. **Azure CLI मार्फत लगइन गर्नु भएको छ** — आफ्नो टर्मिनलमा `az login` चलाउनुहोस्।
3. **आवश्यक वातावरण चर सेट गरिएको छ:**
   - `AZURE_AI_PROJECT_ENDPOINT` — तपाईको Azure AI Foundry प्रोजेक्टको अन्त्यबिन्दु।
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — तपाईले डिप्लोय गरेको मोडेलको नाम।

तलको सेलले तपाईलाई आवश्यक पायथन प्याकेजहरू इन्स्टल गर्छ।


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## तपाइँको पहिलो एजेन्ट सिर्जना गर्दै

एजेन्टलाई दुई कुरा आवश्यक हुन्छ:

- **निर्देशनहरू** जसले यसलाई *को हो* र *कसरी व्यवहार गर्ने* (एक प्रणाली प्राम्प्ट) बताउँछन्।
- **उपकरणहरू** — `@tool` ले सजाइएको Python कार्यहरू जुन एजेन्टले जानकारी प्राप्त गर्न वा क्रियाहरू गर्न कल गर्न सक्छ।

तल हामीले एक साधारण उपकरण परिभाषित गरेका छौं जसले लोकप्रिय vacation गन्तव्यहरूको सूची फर्काउँछ। एक प्रयोगकर्ताले यात्रा सिफारिसहरूको लागि सोध्दा एजेन्टले यस उपकरणको उपयोग गर्नेछ।


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## स्ट्रिमिङ प्रतिक्रियाहरू

अधिक अन्तरक्रियात्मक अनुभवको लागि तपाईं एजेन्टको प्रतिक्रियालाई **स्ट्रिम** गर्न सक्नुहुन्छ। पूर्ण उत्तर कुर्नुको सट्टा, एजेन्टले उत्पन्न भैसकेका पाठ टुक्राहरूलाई क्रमशः प्रदान गर्दछ। यो च्याट इन्टरफेसहरूमा विशेष गरी उपयोगी हुन्छ जहाँ तपाईं तत्काल आउटपुट प्रदर्शन गर्न चाहनुहुन्छ।


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## सारांश

यस पाठमा तपाईंले के सिक्नुभयो:

- **प्रोभर सिर्जना गर्नुहोस्** जुन `AzureAIProjectAgentProvider` मार्फत Azure AI Foundry Agent Service सँग जडान हुन्छ।
- **उपकरण परिभाषित गर्नुहोस्** `@tool` डेकोरेटर प्रयोग गरेर ताकि एजेन्टले तपाईंका Python फंक्शनहरू कल गर्न सकून्।
- **एजेन्ट चलाउनुहोस्** प्रयोगकर्ताको सन्देशसहित र यसको प्रतिक्रिया प्रिन्ट गर्नुहोस्।
- **प्रतिक्रियाहरू स्ट्रिम गर्नुहोस्** वास्तविक-समय आउटपुटका लागि।

अर्को पाठमा हामी एजेन्टिक फ्रेमवर्कहरू थप गहिराइमा अन्वेषण गर्नेछौं र एजेन्टहरूलाई थप शक्तिशाली उपकरणहरू र बहु-चरणीय तर्क क्षमताहरू कसरी दिने भन्ने सिक्नेछौं।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
